# LLMSlim round-3 benchmark builds: `llmslim_v4`

This notebook builds LLMSlim **v4** (PR #51: messages with blank lines stay whole in transcripts), chosen on the dev splits only after the round-3 results were published, on the suites it changes: LoCoMo (1,540 questions; lexical at 1,000/2,500/10,000 tokens, reranker at 1,000/2,500) and LongMemEval-S (180; lexical and reranker at 2,500/10,000). NaturalQuestions and HotpotQA output is byte-identical to v3.

It **never calls a model API**. It only produces the exact context each system would send to the reader, so anyone can rerun it and compare the outputs byte for byte. The reader (Claude Haiku 5.5) and the judge run on these published contexts afterwards.

- **Code:** [Thanatos9404/llmslim](https://github.com/Thanatos9404/llmslim) at commit `513d9110eed041ab87355ac381cb91d218e661a6`, `benchmarks/live/round3/`. Splits were committed before any result: `splits_r3.json`.
- **Data:** downloaded from each dataset's original source (lost-in-the-middle, HotpotQA, LoCoMo, LongMemEval) and checked by sha256.
- **Environments:** pinned Python 3.12 venvs (`kaggle_run.py setup`), separate from Kaggle's image. LLMSlim's default install (no NLTK).
- **Output:** `round3_llmslim_v4.zip`: one JSONL per system and suite (`id`, `context`, `tokens`, `build_ms`) and `manifest-llmslim_v4.json` (commit, GPUs, package versions, data and output hashes).

Settings: accelerator **GPU T4 x2** (the reranker arms), internet **on**. Runtime: under an hour. A rerun resumes from the cases already built.

In [ ]:
import subprocess
import sys

COMMIT = "513d9110eed041ab87355ac381cb91d218e661a6"
JOB = "llmslim_v4"
R = "/tmp/src/benchmarks/live/round3/kaggle_run.py"
subprocess.run(
    "rm -rf /tmp/src && git init -q /tmp/src && cd /tmp/src && git remote add origin https://github.com/Thanatos9404/llmslim && git fetch -q --depth 1 origin "
    + COMMIT
    + " && git checkout -q FETCH_HEAD && git log -1 --format='%H %s'",
    shell=True,
    check=True,
)
subprocess.run([sys.executable, R, "setup", "--work", "/tmp/r3"], check=True)
subprocess.run([sys.executable, R, "data", "--work", "/tmp/r3"], check=True)
subprocess.run([sys.executable, R, "build", "--work", "/tmp/r3", "--job", JOB], check=True)
subprocess.run(
    [
        sys.executable,
        R,
        "package",
        "--work",
        "/tmp/r3",
        "--job",
        JOB,
        "--zip",
        "/kaggle/working/round3_" + JOB + ".zip",
    ],
    check=True,
)